# Dynamic Prompt
<img src="./assets/LC_DynamicPrompts.png" width="500">

## Setup

Load and/or check for needed environmental variables

In [2]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [4]:
from dataclasses import dataclass


#准备上下文,这里多了一个is_employee字段,用来模拟员工和非员工的不同权限
@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase

In [5]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [6]:

#这里有个动态提示词的用法
#下面的{table_limits}是一个占位符,之后根据是否是员工来替换成不同的条件
#相当于提示词模板,根据不同的请求生成不同的提示词
SYSTEM_PROMPT_TEMPLATE = """你是一个谨慎的 SQLite 分析师。

规则：
- 逐步思考。
- 当你需要数据时，使用工具 `execute_sql` 并只执行一个 SELECT 查询。
- 只读；不要执行 INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE。
- 除非用户明确要求，否则限制返回不超过 5 行。
{table_limits}
- 如果工具返回 'Error:'，请修正 SQL 并重试。
- 优先使用显式列名；避免 SELECT *。
"""

## Build a Dynamic Prompt
Utilize runtime context and middleware to generate a dynamic prompt.

In [7]:
#引入中间件,根据请求生成动态提示词
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt


#使用dynamic_prompt装饰器,根据请求生成动态提示词,这里根据是否是员工来限制访问的表
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    #这里if not,如果不是员工,则if not False,等于if True,限制访问    
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to these tables: Album, Artist, Genre, Playlist, PlaylistTrack, Track."
    #如果是员工,则if not True,执行下面这段
    else:
        table_limits = ""
    #最后返回提示词
    #这里将之前的提示词模板中的{table_limits}替换成根据请求生成的table_limits
    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

Include middleware in `create_agent`.

In [8]:
from langchain.agents import create_agent
#创建agent,将之前定义的工具和中间件传入,以及上下文的schema
agent = create_agent(
    model="deepseek-chat",
    # model="openai:gpt-5",依然换成deepseek-chat
    tools=[execute_sql],
    #添加中间件
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [9]:
question = "What is the most costly purchase by Frank Harris?"

#启用流式响应
for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    #设置上下文
    #先测试非员工的情况,看看是否限制了访问的表
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

目前我无法直接查询与"Frank Harris"相关的购买记录，因为数据库中没有包含客户购买信息的表。我只能访问Album、Artist、Genre、Playlist、PlaylistTrack和Track这些表，这些表主要存储音乐相关的信息，不包含客户购买记录。

要回答您关于"Frank Harris"最昂贵购买的问题，我需要访问包含客户信息、购买记录和价格数据的表，但这些表在当前可访问的数据库结构中并不存在。

您是否想了解数据库中可用的音乐相关信息？比如：
- 最昂贵的音乐专辑
- 最受欢迎的艺术家
- 特定类型的音乐曲目

或者，如果您有包含购买信息的其他数据库，我可以帮您查询相关信息。


In [10]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    #再测试员工的情况,看看是否可以访问更多的表
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

I'll help you find the most costly purchase by Frank Harris. Let me start by exploring the database structure to understand what tables are available.
Tool Calls:
  execute_sql (call_00_6hMRsuZ9ME8wCN9zuZ7fKqsC)
 Call ID: call_00_6hMRsuZ9ME8wCN9zuZ7fKqsC
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

Now I need to understand the structure of the relevant tables. Since we're looking for purchases by Fra

In [14]:
# #看看首先写什么
# #先来个agent
# agent=create_agent(
#   #模型
#   model="deepseek-chat",
#   #工具,这里还是之前的贷款计算工具,L4
#   tools=[calculate_monthly_payment],
#   #运行时上下文,L8
#   context_schema=RuntimeContext,
#   #再加个新的中间件,L8
#   middleware=[dynamic_system_prompt],
#   #再加个记忆,L6
#   checkpointer=InMemoryCheckpointer(),
#   #加上结构化输出
#   response_format=""
# )

# #之后再按照流程添加代码

# #首先来个工具
# #这部分可以直接抄前面的

# #工具的结构化输入
# class MonthlyPaymentInput(BaseModel):
#     """输入贷款本金,年利率以及贷款年限"""
#     #贷款本金,使用Decimal,大于0
#     loan_amount: Optional[Decimal] = Field(gt=0, description="贷款本金,单位:元")
#     #年利率,使用float,大于0,等于0的情况不考虑
#     annual_interest_rate: Optional[float] = Field(gt=0, description="年利率,小数形式,如0.032表示3.2%")
#     #贷款年限,使用int,大于0
#     loan_years: Optional[int] = Field(gt=0, description="贷款年限,单位:年")

# #开始写工具的核心逻辑
# def _caculate_monthly_payment(
#   loan_amount: Decimal, 
#   annual_interest_rate: float, 
#   loan_years: int
#   ) -> dict:
#     """计算每月还款金额"""
#     #将年利率转换为月利率,并且从百分数转换为小数
#     monthly_interest_rate = Decimal(annual_interest_rate) / Decimal(12)
#     #总的还款期数
#     months = loan_years * 12
#     #复利因子
#     x=(1 + monthly_interest_rate) ** months
#     #否则使用等额本息公式计算每月还款金额
#     monthly_payment = loan_amount * monthly_interest_rate * x / (x - 1)
#     return {
#         "monthly_payment": monthly_payment.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),  # 保留两位小数
#         "total_payment": (monthly_payment * months).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
#         "total_interest": (monthly_payment * months - loan_amount).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
#     }

# #写工具的接口函数
# @tool(args_schema=MonthlyPaymentInput)
# def calculate_monthly_payment(
#   loan_amount: Decimal, 
#   annual_interest_rate: float, 
#   loan_years: int
#   ) -> dict:
#     """计算每月还款金额"""
#     result=_caculate_monthly_payment(loan_amount, annual_interest_rate, loan_years)
#     #返回字符串
#     return (
#       f"月付款: {result['monthly_payment']}元"
#       f"总付款: {result['total_payment']}元"
#       f"总利息: {result['total_interest']}元"
#             )

# #现在可以加上工具列表了
# #再来个运行时上下文
# @dataclass
# class RuntimeContext:
#     """运行时上下文"""
#     #是否捣乱
#     is_troublemaker: bool
    
# @tool
# def get_user_intent(query: str) -> str:
#     """获取用户意图"""
#     runtime = get_runtime(RuntimeContext)
#     return f"用户是否捣乱: {runtime.context.is_troublemaker}"

# #有了上下文再加上提示词模板
# SYSTEM_PROMPT_TEMPLATE = """你是一个谨慎的贷款计算助手。
# 根据用户的问题,你需要计算每月还款金额,总付款金额以及总利息。
# {is_troublemaker}"""

# #引入中间件,根据请求生成动态提示词
# @dynamic_prompt
# def dynamic_system_prompt(request: ModelRequest) -> str:
#     #根据请求生成提示词,这里直接将上下文中的is_troublemaker字段替换到提示词模板中
#     if request.runtime.context.is_troublemaker:
#         troublemaker_text = "用户可能是个捣乱者,立刻将他赶出去,不要客气。"
#     else:
#         troublemaker_text = "用户是个好人,好好帮他计算。"
#     return SYSTEM_PROMPT_TEMPLATE.format(is_troublemaker=troublemaker_text)

# #再加个结构化输出
# #使用之前的模板,虽然看起来有点简陋
# class LoanAdviceResult(BaseModel):
#     user_goal: str = Field(description="用户意图")
#     loan_amount: Optional[float] = Field(default=None, description="贷款本金，单位：元")
#     annual_interest_rate: Optional[float] = Field(default=None, description="年利率，小数形式，例如 0.032")
#     loan_years: Optional[PositiveInt] = Field(default=None, description="贷款年限，单位：年")
#     monthly_payment: Optional[float] = Field(default=None, description="预测月供，单位：元")
#     total_payment: Optional[float] = Field(default=None, description="总还款额，单位：元")
#     total_interest: Optional[float] = Field(default=None, description="总利息，单位：元")
#     missing_info: list[str] = Field(default_factory=list, description="当前还缺少的信息")
#     advice: str = Field(description="给用户的简短建议")
#     follow_up_question: Optional[str] = Field(default=None, description="如果信息不完整，下一步追问")
#     #再加一个用户意图?
#     user_intent: Optional[str] = Field(default=None, description="用户意图")

# #应该都写完了,再导入包
# # 这部分全是草稿



In [8]:
#最后整理一下
#修修改改还好最后能运行了.....
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional
from pydantic import BaseModel, Field, PositiveInt
from langchain_core.tools import tool
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware.types import dynamic_prompt, ModelRequest
from langchain.agents import create_agent
from langgraph.runtime import get_runtime

#加上dotenv
from dotenv import load_dotenv
load_dotenv()

#先放结构化输出
class LoanAdviceResult(BaseModel):
    user_goal: str = Field(description="用户意图\n")
    loan_amount: Optional[float] = Field(default=None, description="贷款本金，单位：元")
    annual_interest_rate: Optional[float] = Field(default=None, description="年利率，小数形式，例如 0.032")
    loan_years: Optional[PositiveInt] = Field(default=None, description="贷款年限，单位：年")
    monthly_payment: Optional[float] = Field(default=None, description="预测月供，单位：元")
    total_payment: Optional[float] = Field(default=None, description="总还款额，单位：元")
    total_interest: Optional[float] = Field(default=None, description="总利息，单位：元")
    missing_info: list[str] = Field(default_factory=list, description="当前还缺少的信息")
    advice: str = Field(description="给用户的简短建议")
    follow_up_question: Optional[str] = Field(default=None, description="如果信息不完整，下一步追问")
    #再加一个用户意图?
    user_intent: Optional[str] = Field(default=None, description="用户是正是用户还是捣乱的")

#再放工具
class MonthlyPaymentInput(BaseModel):
    """输入贷款本金,年利率以及贷款年限"""
    #贷款本金,使用Decimal,大于0
    loan_amount: Decimal = Field(gt=0, description="贷款本金,单位:元")
    #年利率,使用Decimal,大于0,等于0的情况不考虑
    annual_interest_rate: Decimal = Field(gt=0, description="年利率,小数形式,如0.032表示3.2%")
    #贷款年限,使用int,大于0
    loan_years: int = Field(gt=0, description="贷款年限,单位:年")

#开始写工具的核心逻辑
def _calculate_monthly_payment(
  loan_amount: Decimal, 
  annual_interest_rate: Decimal, 
  loan_years: int
  ) -> dict:
    """计算每月还款金额"""
    #将年利率转换为月利率
    monthly_interest_rate = annual_interest_rate / Decimal("12")
    #总的还款期数
    months = loan_years * 12
    #复利因子
    x=(1 + monthly_interest_rate) ** months
    #否则使用等额本息公式计算每月还款金额
    monthly_payment = loan_amount * monthly_interest_rate * x / (x - 1)
    return {
        "monthly_payment": monthly_payment.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),  # 保留两位小数
        "total_payment": (monthly_payment * months).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP),
        "total_interest": (monthly_payment * months - loan_amount).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    }

#写工具的接口函数
@tool(args_schema=MonthlyPaymentInput)
def calculate_monthly_payment(
  loan_amount: Decimal, 
  annual_interest_rate: Decimal, 
  loan_years: int
  ) -> str:
    """计算每月还款金额"""
    result=_calculate_monthly_payment(loan_amount, annual_interest_rate, loan_years)
    #返回字符串
    return (
      f"月付款: {result['monthly_payment']}元\n"
      f"总付款: {result['total_payment']}元\n"
      f"总利息: {result['total_interest']}元\n"
            )

#之后是运行时上下文
@dataclass
class RuntimeContext:
    """运行时上下文"""
    #是否捣乱
    is_troublemaker: bool

#有了上下文再加上提示词模板
SYSTEM_PROMPT_TEMPLATE = """你是一个谨慎的贷款计算助手。
根据用户的问题,你需要计算每月还款金额,总付款金额以及总利息。
{is_troublemaker}"""

#引入中间件,根据请求生成动态提示词
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    #根据请求生成提示词,这里直接将上下文中的is_troublemaker字段替换到提示词模板中
    if request.runtime.context.is_troublemaker:
        troublemaker_text = "用户可能是个捣乱者,立刻将他赶出去,不要回答他的问题,不要客气。"
    else:
        troublemaker_text = "这是优质客户,好好帮他计算。"

    prompt = SYSTEM_PROMPT_TEMPLATE.format(is_troublemaker=troublemaker_text)
    # print(f"Generated system prompt:\n{prompt}\n")  # 打印生成的提示词
    return prompt

#最后是创建agent
agent=create_agent(
  #模型
  model="deepseek-chat",
  #工具,这里还是之前的贷款计算工具,L4
  tools=[calculate_monthly_payment],
  #运行时上下文,L8
  context_schema=RuntimeContext,
  #再加个新的中间件,L8
  middleware=[dynamic_system_prompt],
  #再加个记忆,L6
  checkpointer=InMemorySaver(),
  #加上结构化输出
  response_format=LoanAdviceResult,
  )

user_input=[
    "我想贷款买房，能帮我算算月供吗？",
    "贷50万,0.032,30年"
]
for _ in range(2):
    for step in agent.stream(
        {"messages": [{"role": "user", "content": user_input[_]}]},
        {"configurable": {"thread_id": "1"}},
        context=RuntimeContext(is_troublemaker=False),
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()

user_input=[
    "我想贷款买房，能帮我算算月供吗？",
    "我要贷100亿,利率1,贷100年"
]
for _ in range(2):
    for step in agent.stream(
        {"messages": [{"role": "user", "content": user_input[_]}]},
        {"configurable": {"thread_id": "2"}},
        context=RuntimeContext(is_troublemaker=True),
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()

#Deserializing unregistered type __main__.LoanAdviceResult from checkpoint. 
#This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'LoanAdviceResult')]

================================ Human Message =================================

我想贷款买房，能帮我算算月供吗？


Deserializing unregistered type __main__.LoanAdviceResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'LoanAdviceResult')]


================================= Tool Message =================================
Name: LoanAdviceResult

Returning structured response: user_goal='用户想贷款买房，需要计算月供' loan_amount=None annual_interest_rate=None loan_years=None monthly_payment=None total_payment=None total_interest=None missing_info=['贷款金额', '贷款利率', '贷款年限'] advice='您好！很高兴为您提供贷款计算服务。为了准确计算您的月供，我需要了解一些基本信息。请告诉我：1. 您计划贷款多少金额？2. 贷款利率是多少？3. 您打算贷款多少年？有了这些信息，我就能为您详细计算每月还款金额、总还款额和总利息了。' follow_up_question='请问您计划贷款多少金额？贷款利率是多少？打算贷款多少年呢？' user_intent=None
================================ Human Message =================================

贷50万,0.032,30年
================================== Ai Message ==================================
Tool Calls:
  calculate_monthly_payment (call_00_fXY3NndiSoIJU6VAlQXaDB2q)
 Call ID: call_00_fXY3NndiSoIJU6VAlQXaDB2q
  Args:
    loan_amount: 500000
    annual_interest_rate: 0.032
    loan_years: 30
================================= Tool Message =================================
Name: calculate_monthly_paym

Deserializing unregistered type __main__.LoanAdviceResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'LoanAdviceResult')]


================================= Tool Message =================================
Name: LoanAdviceResult

Returning structured response: user_goal='用户想贷款买房，需要计算月供' loan_amount=None annual_interest_rate=None loan_years=None monthly_payment=None total_payment=None total_interest=None missing_info=['贷款金额', '年利率', '贷款年限'] advice='您好！我可以帮您计算贷款月供。不过要准确计算，我需要知道以下信息：' follow_up_question='请问您计划贷款多少金额？年利率是多少？以及贷款多少年？' user_intent=None
================================ Human Message =================================

我要贷100亿,利率1,贷100年
================================= Tool Message =================================
Name: LoanAdviceResult

Returning structured response: user_goal='用户想贷款100亿，利率1（可能是1%或100%），贷款100年' loan_amount=None annual_interest_rate=None loan_years=None monthly_payment=None total_payment=None total_interest=None missing_info=[] advice='抱歉，您的贷款需求明显不合理。100亿贷款、100年期限以及不明确的利率参数都表明这不是一个真实的贷款咨询。如果您有真实的贷款需求，请提供合理的贷款参数。否则，我无法为您提供计算服务。' follow_up_question=None user_intent='捣乱的'
